#Install dependencies



In [ ]:
# Force-reinstall compatible versions of numpy, mediapipe, and opencv
!pip install --upgrade --force-reinstall numpy==1.23.5 opencv-python-headless==4.8.0.76 mediapipe==0.10.0


  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached opencv_python_headless-4.8.0.76-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.0 (from versions: 0.10.5, 0.10.7, 0.10.8, 0.10.9, 0.10.10, 0.10.11, 0.10.13, 0.10.14, 0.10.15, 0.10.18, 0.10.20, 0.10.21)
ERROR: No matching distribution found for mediapipe==0.10.0


In [ ]:
#Import libraries
import os
os.system('pip install mediapipe > /dev/null')
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
!pip install torchsummary


#Mount to google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Dataset Path

In [ ]:
dataset_path = '/content/drive/My Drive/hgr/hagriddataset'


#Extracting hand landmarks using mediapipe

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
from tqdm import tqdm

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True)
mp_drawing = mp.solutions.drawing_utils

X = []
y = []

dataset_path = '/content/drive/MyDrive/hgr/hagriddataset/hagrid-sample-30k-384p/hagrid_30k'
classes = sorted(os.listdir(dataset_path))

for label in classes:
    class_dir = os.path.join(dataset_path, label)
    for img_file in tqdm(os.listdir(class_dir), desc=f"Processing {label}"):
        img_path = os.path.join(class_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result = hands.process(img_rgb)
        if result.multi_hand_landmarks:
            landmarks = result.multi_hand_landmarks[0].landmark
            coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks]).flatten()
            X.append(coords)
            y.append(label)

hands.close()

X = np.array(X)
y = np.array(y)

np.save('X_landmarks.npy', X)
np.save('y_labels.npy', y)


#Encode labels ans split the dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = np.load('/content/drive/MyDrive/model1/X_landmarks.npy')
y = np.load('/content/drive/MyDrive/model1/y_labels.npy')

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)


'le = LabelEncoder()\ny_encoded = le.fit_transform(y)\n\nX_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)'

save label encoder

In [ ]:
import pickle
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

create pytorch dataset and dataloaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GestureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = GestureDataset(X_train, y_train)
val_dataset = GestureDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


#Define the neural network model

In [ ]:
import torch.nn as nn
class GestureNet(nn.Module):
    def __init__(self, num_classes):
        super(GestureNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


Train Model 

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GestureNet(num_classes=len(le.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float('inf')
patience = 5
wait = 0
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break


Epoch 1/50, Train Loss: 205.5900, Val Loss: 46.7015
Epoch 2/50, Train Loss: 166.1430, Val Loss: 36.2523
Epoch 3/50, Train Loss: 132.9688, Val Loss: 29.0691
Epoch 4/50, Train Loss: 110.5689, Val Loss: 22.9979
Epoch 5/50, Train Loss: 92.3731, Val Loss: 19.2681
Epoch 6/50, Train Loss: 79.0455, Val Loss: 16.9965
Epoch 7/50, Train Loss: 67.5348, Val Loss: 13.5197
Epoch 8/50, Train Loss: 61.6004, Val Loss: 13.5322
Epoch 9/50, Train Loss: 56.4134, Val Loss: 11.6551
Epoch 10/50, Train Loss: 52.7915, Val Loss: 10.6125
Epoch 11/50, Train Loss: 50.2490, Val Loss: 10.6025
Epoch 12/50, Train Loss: 48.4440, Val Loss: 10.6032
Epoch 13/50, Train Loss: 45.9831, Val Loss: 10.3622
Epoch 14/50, Train Loss: 46.3068, Val Loss: 9.8572
Epoch 15/50, Train Loss: 43.8019, Val Loss: 9.1742
Epoch 16/50, Train Loss: 43.2110, Val Loss: 10.7238
Epoch 17/50, Train Loss: 41.3744, Val Loss: 9.6569
Epoch 18/50, Train Loss: 41.3755, Val Loss: 8.6846
Epoch 19/50, Train Loss: 40.7378, Val Loss: 9.5667
Epoch 20/50, Train Los

#Evaluation


In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
import torch.nn as nn

class GestureNet(nn.Module):
    def __init__(self, num_classes):
        super(GestureNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
import pickle

with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

num_classes = len(le.classes_)

model = GestureNet(num_classes)
model.load_state_dict(torch.load("best_model.pth", map_location=torch.device('cpu')))
model.eval()


GestureNet(
  (net): Sequential(
    (0): Linear(in_features=63, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=6, bias=True)
  )
)

In [ ]:
X_test = np.load("/content/X_landmarks.npy")
y_test = np.load("/content/y_labels.npy")

In [ ]:
y_test_encoded = le.transform(y_test)


In [ ]:

inputs = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    outputs = model(inputs)
    predicted = torch.argmax(outputs, axis=1).numpy()

accuracy = accuracy_score(y_test_encoded, predicted)
print(f"Test Accuracy: {accuracy*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test_encoded, predicted, target_names=le.classes_))


Open camera for testing


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

def capture_image():
    js = '''
    async function capture() {
      const div = document.createElement('div');
      const video = document.createElement('video');
      const canvas = document.createElement('canvas');
      const context = canvas.getContext('2d');
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;

      await new Promise(resolve => setTimeout(resolve, 3000));

      context.drawImage(video, 0, 0);

      stream.getVideoTracks()[0].stop();
      div.remove();

      return canvas.toDataURL('image/jpeg', 0.8);
    }
    capture();
    '''
    # Evaluate JS and get image data URL
    data_url = eval_js(js)

    # Decode base64 image
    header, encoded = data_url.split(",", 1)
    binary_data = b64decode(encoded)

    # Convert to numpy array and then to OpenCV image
    img_array = np.frombuffer(binary_data, dtype=np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    return img


In [ ]:
import mediapipe as mp
import torch

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True,
                       max_num_hands=1,
                       min_detection_confidence=0.7)

def predict_gesture(image, model, label_encoder):
    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        landmarks = results.multi_hand_landmarks[0]
        coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark]).flatten()
        input_tensor = torch.tensor(coords, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            output = model(input_tensor)
            pred_idx = torch.argmax(output, dim=1).item()
            pred_label = label_encoder.inverse_transform([pred_idx])[0]
            return pred_label
    else:
        return "No hand detected"



In [ ]:
img = capture_image()
print("Captured image:")
from google.colab.patches import cv2_imshow
cv2_imshow(img)

gesture_label = predict_gesture(img, model, le)
print(f"Predicted Gesture: {gesture_label}")